In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 80.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## Local Inference on GPU
Model page: https://huggingface.co/casperhansen/llama-3-8b-instruct-awq

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/casperhansen/llama-3-8b-instruct-awq)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 14.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 13.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 9.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  U

In [ ]:
!uv pip install vllm==0.18.0 --torch-backend=auto

Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 3.76s
Prepared 75 packages in 51.55s
Uninstalled 14 packages in 1.01s
Installed 75 packages in 720ms
 + anthropic==0.105.2
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==6.1.1
 + compressed-tensors==0.13.0
 - cuda-bindings==12.9.6
 + cuda-bindings==13.0.3
 - cuda-python==12.9.6
 + cuda-python==13.0.3
 + depyf==0.20.0
 + detect-installer==0.1.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.18.0
 + fastar==0.11.0
 + flashinfer-python==0.6.6
 + gguf==0.19.0
 - huggingface-hub==1.16.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.2
 + model-hosting-container-standards==0.1.15
 + msgspec==0.21.1
 - numba==0.60.0
 + numba==0.61.2
 + nvidia-cu

In [ ]:
import os
import sys
import contextlib
# Required for runtime swap
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Import vLLM's system utils explicitly
import vllm.utils.system_utils

# Create a dummy context manager that skips the fileno() logic
@contextlib.contextmanager
def colab_safe_suppress():
    yield  # Do absolutely nothing

# Neutralize vLLM's native suppress_stdout with our dummy function
vllm.utils.system_utils.suppress_stdout = colab_safe_suppress

In [ ]:
# Library imports
from vllm import LLM, SamplingParams
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
import torch.nn as nn

from vllm.model_executor.layers.layernorm import RMSNorm

In [ ]:
# Set runtime device
DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [ ]:
@triton.jit
def rmsnorm(x_ptr, residual_ptr, gamma_ptr, out_ptr, num_tokens, token_stride, res_stride, hidden_dim, eps, has_residual: tl.constexpr, block_size_m: tl.constexpr, block_size_k: tl.constexpr):
  """
  Kernel which applies RMSNorm to the input tokens.

  Parameters:
  x_ptr: Pointer to the input tokens.
  residual_ptr: Pointer to the residual tokens.
  gamma_ptr: Pointer to the gamma values.
  out_ptr: Pointer to the output tokens.
  num_tokens: Number of tokens.
  token_stride: Stride between tokens.
  res_stride: Stride between residual tokens.
  hidden_dim: Hidden dimension.
  eps: Epsilon value.
  has_residual: Whether residual tokens are present.
  block_size_m: Block size for the number of tokens.
  block_size_k: Block size for the hidden dimension.
  """
  # Get 1D block id
  pid = tl.program_id(axis = 0)

  # Set epsilon value for handling zero embedding vectors
  epsilon = eps

  # Thread id with which block starts
  block_start = pid * block_size_m

  # Get the offsets for the number of tokens for a specific block
  token_offsets = block_start + tl.arange(0, block_size_m)

  # Mask for handling the final block
  token_mask = token_offsets < num_tokens

  # Get the offsets for the hidden dimension. Since the block handles entire token at once, we do not add block start.
  hidden_dim_offsets = tl.arange(0, block_size_k)

  # For handling offsets beyond hidden dimension length
  hidden_dim_mask = hidden_dim_offsets < hidden_dim

  # Load the input tokens and the gamma values
  tokens = tl.load(x_ptr + token_offsets[:, None] * token_stride + hidden_dim_offsets[None, :], mask = token_mask[:, None] * hidden_dim_mask[None, :])
  gammas = tl.load(gamma_ptr + hidden_dim_offsets, mask = hidden_dim_mask)

  # For handling residual
  if has_residual:
    # Load residual
    residual = tl.load(residual_ptr + token_offsets[:, None] * res_stride + hidden_dim_offsets[None, :], mask = token_mask[:, None] * hidden_dim_mask[None, :])

    # Add residual
    tokens += residual

    # Store to be passed back for later block
    tl.store(residual_ptr + token_offsets[:, None] * res_stride + hidden_dim_offsets[None, :], tokens, mask = token_mask[:, None] * hidden_dim_mask[None, :])

  # Upcast from float16 to float32 to prevent underflow
  tokens_f32 = tokens.to(tl.float32)
  gammas_f32 = gammas.to(tl.float32)

  # MS value
  var = tl.sum(tokens_f32 * tokens_f32, axis = 1) / hidden_dim + epsilon

  # Inverse RMS value
  inv_rms_vals = 1.0 / tl.sqrt(var)

  # Normalize the input token matrix
  normed_outputs_f32 = tokens * inv_rms_vals[:, None] * gammas_f32[None, :]

  # Downcast to float16 to match model architecture
  normed_outputs = normed_outputs_f32.to(tokens.dtype)

  # Store the output
  tl.store(out_ptr + token_offsets[:, None] * token_stride + hidden_dim_offsets[None, :], normed_outputs, mask = token_mask[:, None] * hidden_dim_mask[None, :])

In [ ]:
def custom_RMSNorm_cuda_forward(self, x, residual: torch.Tensor | None = None):
  """Function to replace the forward method for CUDA RMSNorm in vLLM Llama."""
  if self.variance_size_override is not None:
    return self.forward_native(x, residual)

  # Boolean to indicate prescence of residual
  has_residual = False

  # Store 3d shape for output
  original_shape = x.shape

  # Squash first two dimensions to handle token level dimensionality
  squashed_x = x.view([-1, x.shape[-1]])

  # Initialize dummy residual value
  squashed_residuals = squashed_x

  # For handling residual
  if residual is not None:
    has_residual = True
    squashed_residuals = residual.view([-1, residual.shape[-1]])

  # Get the total number of tokens and the hidden dimension
  total_tokens = squashed_x.shape[0]
  hidden_dim = squashed_x.shape[1]

  # Initialize output matrix
  output = torch.empty((total_tokens, hidden_dim), device = x.device, dtype = x.dtype)

  # For handling cases when hidden dimension is not a perfect power of 2
  block_size_k = triton.next_power_of_2(hidden_dim)

  # Initialize the block grid
  grid = lambda meta: ((triton.cdiv(total_tokens, meta['block_size_m'])),)

  # Initialize epsilon
  eps = self.variance_epsilon

  # Invoke the kernel
  rmsnorm[grid](squashed_x, squashed_residuals, self.weight.data, output, total_tokens, squashed_x.stride(0), squashed_residuals.stride(0), hidden_dim, eps, has_residual, block_size_m = 1, block_size_k = block_size_k)

  # Match return statement according to original function
  if residual is not None:
    return (output.view(original_shape), residual.view(original_shape))

  return output.view(original_shape)

# Invoke runtime swap
RMSNorm.forward_cuda = custom_RMSNorm_cuda_forward

In [ ]:
# Initialize the vLLM object
llm  = LLM(model="casperhansen/llama-3-8b-instruct-awq", quantization = "awq", tensor_parallel_size = 1, enforce_eager = True)

INFO 05-29 19:30:18 [utils.py:233] non-default args: {'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'casperhansen/llama-3-8b-instruct-awq'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

WARNING 05-29 19:30:19 [arg_utils.py:1352] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-29 19:30:42 [model.py:533] Resolved architecture: LlamaForCausalLM
INFO 05-29 19:30:42 [model.py:1582] Using max model len 8192
INFO 05-29 19:30:42 [awq_marlin.py:166] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 05-29 19:30:42 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

INFO 05-29 19:31:39 [weight_utils.py:574] Time spent downloading weights for casperhansen/llama-3-8b-instruct-awq: 51.484668 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-29 19:31:59 [default_loader.py:384] Loading weights took 19.27 seconds
INFO 05-29 19:32:00 [gpu_model_runner.py:4566] Model loading took 5.34 GiB memory and 72.278766 seconds
INFO 05-29 19:32:20 [gpu_worker.py:456] Available KV cache memory: 6.86 GiB
INFO 05-29 19:32:20 [kv_cache_utils.py:1316] GPU KV cache size: 56,224 tokens
INFO 05-29 19:32:20 [kv_cache_utils.py:1321] Maximum concurrency for 8,192 tokens per request: 6.86x
INFO 05-29 19:32:20 [kernel_warmup.py:69] Warming up FlashInfer attention.
INFO 05-29 19:35:07 [core.py:281] init engine (profile, create kv cache, warmup model) took 187.68 seconds
INFO 05-29 19:35:09 [llm.py:391] Supported tasks: ('generate',)


In [ ]:
# Set parameters for the output
params = SamplingParams(temperature=0.8, max_tokens = 100)

# Get the tokenizer for applying chat template
tokenizer = llm.get_tokenizer()

In [ ]:
# Block to fetch prompts from the alpaca dataset
from datasets import load_dataset
ds = load_dataset("tatsu-lab/alpaca", split = "train")
inputs = ds["instruction"][:200]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
formatted_prompts = []
for prompt in inputs:
    # Wrap the raw text in the standard dictionary format
    message = [{"role": "user", "content": prompt}]

    # Apply the template to create the raw string Llama-3 expects
    formatted_string = tokenizer.apply_chat_template(
        conversation=message,
        tokenize=False,
        add_generation_prompt=True
    )

    formatted_prompts.append(formatted_string)

In [ ]:
# Output for the 50 prompt set

# outputs = llm.generate(formatted_prompts, sampling_params = params)

# for output in outputs:
#     prompt = output.prompt
#     generated_text = output.outputs[0].text
#     print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{'role': 'user', 'content': 'What is the capital of Australia?'}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", Generated text: 'The capital of Australia is Canberra.'
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nExplain the theory of relativity in exactly one sentence.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: "According to Albert Einstein's theory of relativity, the laws of physics are the same for all observers, regardless of their relative motion or position in a gravitational field, and the passage of time and the length of objects can vary depending on an observer's frame of reference."
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWho won the World Series in 2016?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'The Chicago Cubs won the World Series in 2016, defeating the Cleveland Ind

In [ ]:
outputs = llm.generate(formatted_prompts, sampling_params = params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

Rendering prompts:   0%|          | 0/200 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/200 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nGive three tips for staying healthy.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'Here are three tips for staying healthy:\n\n1. **Stay Hydrated**: Drink plenty of water throughout the day to help your body function properly. Aim for at least 8 cups (64 ounces) of water daily. You can also consume other hydrating fluids like herbal tea, low-sugar sports drinks, and water-rich foods like fruits and vegetables.\n2. **Get Moving**: Regular exercise is essential for maintaining physical and mental health. Aim for at least 30 minutes of moderate-intensity'
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat are the three primary colors?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'The three primary colors are:\n\n1. **Red**\n2. **Blue**\n3. **Yellow**\n\nThese three colors cannot be created by mixing other colors together, and they ar